## Partindo do que já ficou pronto na Etapa 1 (EDA+Baseline)

Recarregar os dados e repitir o pré-processamento já validado: correção do
TotalCharges, remoção de customerID, encoding das categóricas e split
treino/teste estratificado

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from pipeline import construir_pipeline

df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

df_model = df.drop(columns=['customerID'])
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1109)

## Baseline: Regressão Logística (recapitulando da Etapa 1)

Reproduzir o baseline da Etapa 1, com os dados escalados e
class_weight='balanced', para servir de referência direta na comparação
com os modelos de árvore e rede neural deste notebook.

In [2]:
from sklearn.linear_model import LogisticRegression

lr_model = construir_pipeline(LogisticRegression(max_iter=1000, random_state=1109, class_weight='balanced'))
lr_model.fit(X_train, y_train)

print(classification_report(y_test, lr_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



## Random Forest

Testar um modelo baseado em árvores pra comparar com o baseline linear.
Random Forest costuma lidar melhor com relações não-lineares, e não
precisa dos dados escalados como a regressão logística precisava.

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf_model = construir_pipeline(RandomForestClassifier(class_weight='balanced', random_state=1109))
rf_model.fit(X_train, y_train)

print(classification_report(y_test, rf_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1035
           1       0.56      0.64      0.60       374

    accuracy                           0.77      1409
   macro avg       0.71      0.73      0.72      1409
weighted avg       0.78      0.77      0.78      1409



## Árvore de Decisão (*Decision Tree* / CART)

Testar uma árvore de decisão simples, tende a
ser menos conservadora com a classe minoritária que um *ensemble*.

In [4]:
from sklearn.tree import DecisionTreeClassifier

dt_model = construir_pipeline(DecisionTreeClassifier(class_weight='balanced', random_state=1109))
dt_model.fit(X_train, y_train)

print(classification_report(y_test, dt_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.83      0.80      0.81      1035
           1       0.50      0.54      0.52       374

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.67      1409
weighted avg       0.74      0.73      0.74      1409



## MLPClassifier (rede neural simples)

Testar uma rede neural simples do scikit-learn. 

In [5]:
from sklearn.neural_network import MLPClassifier

mlp_model = construir_pipeline(MLPClassifier(random_state=1109, max_iter=500))
mlp_model.fit(X_train, y_train)

print(classification_report(y_test, mlp_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.83      0.88      0.85      1035
           1       0.60      0.50      0.54       374

    accuracy                           0.78      1409
   macro avg       0.71      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



## Ajuste do MLP: mais iterações + balanceamento manual

MLPClassifier não tem class_weight, mas aceita sample_weight no fit,
calculando pesos balanceados manualmente com compute_sample_weight, igual
em espírito ao que fizemos nos outros modelos. Também aumento max_iter
para resolver o warning de convergência.

In [6]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

mlp_model = construir_pipeline(MLPClassifier(random_state=1109, max_iter=1000))
mlp_model.fit(X_train, y_train, modelo__sample_weight=sample_weights)

print(classification_report(y_test, mlp_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.87      0.75      0.80      1035
           1       0.50      0.69      0.58       374

    accuracy                           0.73      1409
   macro avg       0.68      0.72      0.69      1409
weighted avg       0.77      0.73      0.74      1409



## MLP com balanceamento (SMOTE)

Sem balanceamento, o MLP ficou com recall baixo (0,50), parecido com o
problema que tivemos no baseline original. Com sample_weight subiu para
0,69. Testar agora uma alternativa: oversampling com SMOTE.

In [7]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

base = construir_pipeline(MLPClassifier(random_state=1109, max_iter=1000))
mlp_model = ImbPipeline([
    ('pre', base.named_steps['pre']),
    ('smote', SMOTE(random_state=1109)),
    ('modelo', base.named_steps['modelo']),
])
mlp_model.fit(X_train, y_train)

print(classification_report(y_test, mlp_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.84      0.82      0.83      1035
           1       0.53      0.57      0.55       374

    accuracy                           0.75      1409
   macro avg       0.68      0.69      0.69      1409
weighted avg       0.76      0.75      0.75      1409



## Validação cruzada (Regressão Logística)

Rodar validação cruzada estratificada (5 folds) no modelo campeão, pra
confirmar que o recall de 0,79 não foi só sorte de um split específico, 
e sim um resultado consistente do modelo.

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1109)

final_model = construir_pipeline(LogisticRegression(max_iter=1000, random_state=1109, class_weight='balanced'))
scores = cross_val_score(final_model, X, y, cv=skf, scoring='recall')

print("Recall por fold:", scores)
print("Recall médio:", scores.mean())
print("Desvio padrão:", scores.std())


Recall por fold: [0.8315508  0.82887701 0.77807487 0.80160858 0.78342246]
Recall médio: 0.8047067425556623
Desvio padrão: 0.0222561509521931


## Conclusão da Etapa 2 e Tabela Comparativa de modelos

| Modelo | Recall (classe 1) | Precision (classe 1) | F1-score | Accuracy |
|---|---|---|---|---|
| Regressão Logística (balanced) | **0,79** | 0,52 | 0,63 | 0,75 |
| Random Forest (balanced) | 0,64 | 0,56 | 0,60 | 0,77 |
| Árvore de Decisão (balanced) | 0,54 | 0,50 | 0,52 | 0,73 |
| MLP (sem balanceamento) | 0,50 | 0,60 | 0,54 | 0,78 |
| MLP (sample_weight balanced) | 0,69 | 0,50 | 0,58 | 0,73 |
| MLP (SMOTE) | 0,57 | 0,53 | 0,55 | 0,75 |

Foi testado Regressão Logística, Árvore de Decisão, Random Forest e MLP (sem
balanceamento, com sample_weight e com SMOTE). A Regressão Logística com class_weight='balanced'
venceu em recall por boa margem, e a validação cruzada confirmou que o
resultado é estável.

Modelos mais complexos (árvore, ensemble, rede neural) não superaram o
baseline linear. plausível dado que a EDA já mostrava relações bem
diretas entre variáveis como Contract e tenure com o churn, favorecendo
um modelo linear.

**Modelo campeão: Regressão Logística com class_weight='balanced'**
(validado com recall médio de 0,80 ± 0,022 em 5-fold cross-validation)

In [9]:
import joblib

joblib.dump(lr_model, '../models/churn_pipeline.pkl')

['../models/churn_pipeline.pkl']

In [10]:
loaded_model = joblib.load('../models/churn_pipeline.pkl')

print(loaded_model)
print(loaded_model.named_steps['modelo'].coef_)  # os pesos que o modelo aprendeu pra cada variável
print(loaded_model.named_steps['pre'].named_transformers_['num'].mean_)  # a média que o scaler guardou de cada variável numérica

Pipeline(steps=[('pre',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num', StandardScaler(),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'T